<a href="https://colab.research.google.com/github/rastri-dey/Ground-up-implementations-ML-algorithms-/blob/main/notebooks/RNN_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Description

Building a language model for text generation

**ML Algorithm**: RNN <br>
**Dataset**: Book by H G Wells "The Time Machine" <br>
**Framework**: PyTorch

**Note:** The data loading, vocab class, train code is similar to: [RNN from Scratch](https://colab.research.google.com/github/rastri-dey/Ground-up-implementations-ML-algorithms-/blob/main/notebooks/RNN_scratch_pytorch.ipynb?authuser=1#scrollTo=CwXI-m9xTMFD). The main RNN class uses the built in PyTorch modules.

1.   Import Libraries
2.   Download the dataset
3. Data Processing (Conversion to tokens, create vocab class dictionary)
4. Batch creation (Sequential/ Random DataLoader - to be accepted by PyTorch modules)
5. Create RNN class (no. of layers, hidden states, time steps, inputs)
6. Train model (batch data, model prediction, loss, optimizer)




In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt

import random
import re
import collections
import os
import requests
import hashlib
import math
import time
import numpy as np

In [2]:
# Download method is taken from another notebook
# If there is an existing already downloaded text, no need to learn this

def download(name, cache_dir=os.path.join("..", "data")):
    """Download a file inserted into DATA_HUB, return the local filename."""
    assert name in DATA_HUB, f"{name} does not exist in {DATA_HUB}."
    url, sha1_hash = DATA_HUB[name]
    os.makedirs(cache_dir, exist_ok=True)
    fname = os.path.join(cache_dir, url.split("/")[-1])
    if os.path.exists(fname):
        sha1 = hashlib.sha1()
        with open(fname, "rb") as f:
            while True:
                data = f.read(1048576)
                if not data:
                    break
                sha1.update(data)
        if sha1.hexdigest() == sha1_hash:
            return fname  # Hit cache
    print(f"Downloading {fname} from {url}...")
    r = requests.get(url, stream=True, verify=True)
    with open(fname, "wb") as f:
        f.write(r.content)
    return fname

In [3]:
def read_text():
  '''
  Inputs: Text Book - Book by H G Wells "The Time Machine"
  Outputs: List of strings where each string is Cleaned-up lowercase corresponding to a line from the input file
  Process: Remove any character that is not (^) A-Z and a-z, .strip() removes leading trailing whitespaces, newline character and make all english characters lower
  '''
  with open(download("time_machine"), "r") as f:
    lines = f.readlines()
  return [re.sub("[^A-Za-z]+"," ", line).strip().lower() for line in lines]

def tokenize(lines, token="word"):
  '''
  Inputs: List of strings (where each string is one line from the book)
  Outputs: List of List of all tokens like [['the', 'time', 'machine'], ['by', 'h', 'g', 'wells']]- words or characters (All the words or chars used in the book)
  '''
  if (token == "word"):
    return [line.split() for line in lines]
  elif (token == "char"):
    return [list(line) for line in lines]
  else:
    print("Error: Unknown token type: " + token)

def count_corpus(tokens):
  '''
  Inputs: A 2D list of tokens
  Outputs: A dictionary object of all tokens and their frequencies from the entire book
  Process: Flatten 1D list of all tokens -> Count the frequency of each token through the collections.Counter object which in itself is a dictionary
  '''
  if len(tokens)==0 or isinstance(tokens[0], list):
    tokens_flat = [token for line in tokens for token in line] # Single List of all tokens
  return collections.Counter(tokens_flat)

In [4]:
class Vocab:
  '''
  Create a Vocab class, which assigns a index to each token: It is a dictionary of token to index and index to token
  When vocab is called with a token like Vocab(token) it would return the index of that token
  By Design the Vocab dictionary of token and index is in max to min frequency order, so by index highest frequency tokens appears before the lower frequency tokens
  '''
  def __init__(self, tokens=None, min_freq=0, reserved_tokens=None):
    if tokens == None:
      tokens = []
    if reserved_tokens == None:
      reserved_tokens = []
    counter = count_corpus(tokens)
    freq_tokens = sorted(counter.items(), key = lambda freq: freq[1], reverse=True)
    self.unk_ind, unk_tokens = 0, ["<unk>"] + reserved_tokens
    unk_tokens += [token for token, freq in freq_tokens if freq>=min_freq and token not in unk_tokens]
    self.token_idx, self.idx_token = dict(), []
    for token in unk_tokens:
      self.idx_token.append(token)
      self.token_idx[token] = len(self.idx_token) - 1

  def __len__(self):
      return len(self.idx_token)

  def __getitem__(self, tokens):
    '''
    Returns a list of token indices corresponding to the token list (If the token list is 2D, this list is 2D)
    '''
    if not isinstance(tokens, (list, tuple)):
        return self.token_idx.get(tokens, self.unk_ind)   # If there is no index for that token return 0
    return [self.__getitem__(token) for token in tokens]

  def to_tokens(self, ids):
    '''
    Returns a list of token corresponding to the token indices (If the token indices list is 2D, this list is 2D)
    '''
    if not isinstance(ids, (list, tuple)):
        return self.idx_token[ids]
    return [self.idx_token[id] for id in ids]


In [5]:
def load_corpus_data(max_tokens=-1):
    '''
    Return a single list of token indices corresponding to the book and create a vocabulary from the book
    '''
    lines = read_text()              # List of strings (where each string is one line from the book)
    tokens = tokenize(lines, "char") # 2D List of tokens (Inner 1D list of tokens is each line from the book)
    vocab = Vocab(tokens)            # Create an instance (or object) of Vocab class

    corpus = [vocab[token] for line in tokens for token in line]

    if max_tokens > 0:
        corpus = corpus[:max_tokens]

    return corpus, vocab

In [6]:
def seq_random_sampl(corpus, num_steps, batch_size):
  '''
  Inputs: The textbook data
  Outputs: Batch data: Input data X(sequence of characters) and corresponding labels Y(expected next sequence of characters, given last input)
  Process: Random sampling of sequences of data
  '''
  corpus = corpus[random.randint(0,num_steps-1):] # Based on the Book details, we need the corpus to start from different random starting points
  num_seqs = (len(corpus)-1)//num_steps
  initial_indices = list(range(0, num_seqs*num_steps, num_steps))
  random.shuffle(initial_indices)

  num_batches = num_seqs//batch_size

  def data(pos):
    return corpus[pos:pos+num_steps]

  for i in range(num_batches):
    rand_batch_indices = initial_indices[i:i+batch_size]
    X = [data(pos) for pos in rand_batch_indices]
    Y = [data(pos+1) for pos in rand_batch_indices]
    yield torch.tensor(X), torch.tensor(Y)

In [7]:
def seq_sequential_sampl(corpus, num_steps, batch_size):
  '''
  Inputs: The textbook data
  Outputs: Batch data: Input data X(sequence of characters) and corresponding labels Y(expected next sequence of characters, given last input)
  Process: Sequential sampling of sequences of data
  '''
  offset = random.randint(0, num_steps-1)

  num_tokens = ((len(corpus)-offset-1)//batch_size)*batch_size  # Intention is to make the num_tokens a multiple of batch size, so that matrix is even, -1 is done to consider for the final label char of final input char

  Xs = torch.tensor(corpus[offset:offset+num_tokens])     # A list
  Ys = torch.tensor(corpus[offset+1:offset+num_tokens+1]) # A list
  Xs = Xs.reshape(batch_size, -1)                  # Matrix would be even, because of multiple of num_tokens calculation
  Ys = Ys.reshape(batch_size, -1)

  num_batches = Xs.shape[1]//num_steps

  for i in range(num_batches):
    X = Xs[:, i : i+num_steps]
    Y = Ys[:, i : i+num_steps] # No need of doing pos+1, since its already taken in tensor list Ys
    yield X, Y

In [8]:
class DataLoader:
    '''Create your own DataLoader for loading batches of (Inputs, Labels) iteratively'''

    def __init__(self, batch_size, num_steps, random_sampl, max_tokens):
        if random_sampl:
            self.data_iter_fn = seq_random_sampl
        else:
            self.data_iter_fn = seq_sequential_sampl
        self.corpus, self.vocab = load_corpus_data(max_tokens)
        self.batch_size, self.num_steps = batch_size, num_steps

    def __iter__(self):
        return self.data_iter_fn(self.corpus, self.num_steps, self.batch_size)

In [9]:
# Not learning the textbook data download part
DATA_HUB = dict()
DATA_URL = "http://d2l-data.s3-accelerate.amazonaws.com/"
DATA_HUB["time_machine"] = (DATA_URL + "timemachine.txt", "090b5e7e70c295757f55df93cb0a180b9691891a")

batch_size, num_steps = 32, 35
'''
Process: If we iterate train_iter in a for loop, it would take the __iter__ method from the DataLoader class
and iterate the sequential functions, which in return would yield (X,Y) in batches
'''
train_iter = DataLoader(batch_size, num_steps, random_sampl=False, max_tokens=-1) # Create an instance of DataLoader class
vocab = train_iter.vocab


In [10]:
# Gradient Clipping
# PyTorch computes the gradients of the loss with respect to each of the parameters and stores them in their respective .grad attributes
def grad_clip(theta, model):
  params = model.parameters()
  norm = torch.sqrt(sum(torch.sum((p.grad**2)) for p in params)) # Normalized value over all weights & biases: W_xh, W_hh, b_h, W_hq, b_q
  if norm > theta:
    for param in params:
      param.grad[:] *= theta/norm # theta is the threshold


In [11]:
def predict(model, prefix, num_preds, device):
  '''
  Input: We need a model to predict for the prefix string upto num_preds after the prefix string in the same device where, state, parameters, inputs are
  Output: Returns the prediction upto num_preds step
  Process: This is just a prediction function, model training doesn't happen here.
  We train the model in batches, but prediction happens 1 data per 1 batch, so we initialize batch_size=1 and we predict 1 data per time step
  '''
  state = model.init_state(batch_size=1, device=device)
  outputs = [vocab[prefix[0]]]
  get_input = lambda: torch.tensor([outputs[-1]], device=device).reshape(1,1) # We expect the forward fn to have the input data as a tensor in batch like (N,T) format
  for i in prefix[1:]:
    _, state = model(get_input(),state) # We do not want to pass any argument to the lambda function, it just takes the last character of output
    outputs.append(vocab[i]) # We call the inputs as the outputs here, because we keep the same outputs as the prefix and then start the prediction after prefix ends. So the total output is prefix + new prediction uptil num_preds
  for _ in range(num_preds):
    y, state = model(get_input(),state)        # y is in shape (N*T, D) = (1*1, D)
    outputs.append(int(y.argmax(dim=1).reshape(1))) # We want the max index

  return "".join(vocab.to_tokens(outputs))

In [23]:
# We have taken this part from another notebook, we don't learn this
class Animator:
    """For plotting data in animation."""

    def __init__(
        self,
        xlabel=None,
        ylabel=None,
        legend=None,
        xlim=None,
        ylim=None,
        xscale="linear",
        yscale="linear",
        fmts=("-", "m--", "g-.", "r:"),
        nrows=1,
        ncols=1,
        figsize=(3.5, 2.5),
    ):
        # Incrementally plot multiple lines
        if legend is None:
            legend = []
        display.set_matplotlib_formats("svg")
        self.fig, self.axes = plt.subplots(nrows, ncols, figsize=figsize)
        if nrows * ncols == 1:
            self.axes = [
                self.axes,
            ]
        # Use a lambda function to capture arguments
        self.config_axes = lambda: set_axes(self.axes[0], xlabel, ylabel, xlim, ylim, xscale, yscale, legend)
        self.X, self.Y, self.fmts = None, None, fmts

    def add(self, x, y):
        # Add multiple data points into the figure
        if not hasattr(y, "__len__"):
            y = [y]
        n = len(y)
        if not hasattr(x, "__len__"):
            x = [x] * n
        if not self.X:
            self.X = [[] for _ in range(n)]
        if not self.Y:
            self.Y = [[] for _ in range(n)]
        for i, (a, b) in enumerate(zip(x, y)):
            if a is not None and b is not None:
                self.X[i].append(a)
                self.Y[i].append(b)
        self.axes[0].cla()
        for x, y, fmt in zip(self.X, self.Y, self.fmts):
            self.axes[0].plot(x, y, fmt)
        self.config_axes()
        display.display(self.fig)
        display.clear_output(wait=True)


class Timer:
    """Record multiple running times."""

    def __init__(self):
        self.times = []
        self.start()

    def start(self):
        """Start the timer."""
        self.tik = time.time()

    def stop(self):
        """Stop the timer and record the time in a list."""
        self.times.append(time.time() - self.tik)
        return self.times[-1]

    def avg(self):
        """Return the average time."""
        return sum(self.times) / len(self.times)

    def sum(self):
        """Return the sum of time."""
        return sum(self.times)

    def cumsum(self):
        """Return the accumulated time."""
        return np.array(self.times).cumsum().tolist()


class Accumulator:
    """For accumulating sums over `n` variables."""

    def __init__(self, n):
        self.data = [0.0] * n

    def add(self, *args):
        self.data = [a + float(b) for a, b in zip(self.data, args)]

    def reset(self):
        self.data = [0.0] * len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


def set_axes(axes, xlabel, ylabel, xlim, ylim, xscale, yscale, legend):
    """Set the axes for matplotlib."""
    axes.set_xlabel(xlabel)
    axes.set_ylabel(ylabel)
    axes.set_xscale(xscale)
    axes.set_yscale(yscale)
    axes.set_xlim(xlim)
    axes.set_ylim(ylim)
    if legend:
        axes.legend(legend)
    axes.grid()


def sgd(params, lr, batch_size):
    """Minibatch stochastic gradient descent."""
    with torch.no_grad():
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()

In [24]:
def train_epoch(data_loader, model, loss_fn, optimizer, use_random_sampl, device):
  '''
  We detach state (tuples of state), to avoid gradient tracking on this state tensor calculation
  During sequential sampling, we want to pass the state from one mini-batch to the next. But we don't want to track the gradient changes on this,
  as this would lead to accumulation of gradients over long sequences of data which would lead to issues like exploding or vanishing gradients
  '''
  state, timer = None, Timer()
  metric = Accumulator(2)                # Sum of training loss, no. of tokens
  for X, Y in data_loader:
    if state is None or use_random_sampl:
      state = model.init_state(batch_size=X.shape[0], device=device)
    else:
      if isinstance(model, nn.Module) and not isinstance(state, tuple): # State is not a tuple for nn.RNN
        state.detach_()


    X = X.to(device = device)            # The data reshaping is internally handled within the model
    Y = Y.T.reshape(-1)                  # [N,T] -> [T,N] -> [N*T]
    Y = Y.to(device = device)
    Y_pred, state = model(X, state)      # State is getting transferred from one batch to the next
    l = loss_fn(Y_pred, Y.long()).mean() # Make the loss scalar: Avg loss of one batch
    optimizer.zero_grad()
    l.backward()                         # Cross-entropy loss: backward calculation through time sequence of data
    grad_clip(1, model)                  # Update the gradients if necessary before optimizer.step
    optimizer.step()

    metric.add(l * Y.numel(), Y.numel()) # accumulation of Avg Training loss (per batch) for all batch for all tokens, accumulation of No. of tokens (per batch)
  return math.exp(metric[0] / metric[1]), metric[1] / timer.stop() # Perplexity: exponential of the average cross-entropy loss per token, Training speed


In [25]:
def train(n_epochs, lr, data_loader, model, device, use_random_sampl = False):
  loss_fn = nn.CrossEntropyLoss()
  optimizer = torch.optim.SGD(model.parameters(), lr)
  num_preds = 50
  predict_ = lambda prefix: predict(model, prefix, num_preds, device)
  start_time = time.time()
  for epoch in range(n_epochs):
    ppl, train_speed = train_epoch(data_loader, model, loss_fn, optimizer, use_random_sampl, device)
    if((epoch+1)%10==0):
      print(predict_("time traveller"))

  print(f"Perplexity: {ppl}, Training time: {time.time()-start_time}, Training speed: {train_speed} tokens/sec on device: {device}")

In [26]:
if torch.cuda.is_available():
  device = torch.device('cuda:0')
else:
  device = torch.device('cpu')

In [27]:
vocab_size = len(train_iter.vocab)
print(vocab_size)
print(train_iter.vocab.token_idx)

28
{'<unk>': 0, ' ': 1, 'e': 2, 't': 3, 'a': 4, 'i': 5, 'n': 6, 'o': 7, 's': 8, 'h': 9, 'r': 10, 'd': 11, 'l': 12, 'm': 13, 'u': 14, 'c': 15, 'f': 16, 'w': 17, 'g': 18, 'y': 19, 'p': 20, 'b': 21, 'v': 22, 'k': 23, 'x': 24, 'z': 25, 'j': 26, 'q': 27}


[RNN PyTorch module](https://docs.pytorch.org/docs/stable/generated/torch.nn.RNN.html) <br>
[LSTM PyTorch Module](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html) <br>
[GRU PyTorch Module](https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html)

In [28]:
hidden_size = 256
rnn = nn.RNN(len(vocab), hidden_size) # num_layers = 1
print(rnn.hidden_size, rnn.input_size)

256 28


In [29]:
class RNNModel(nn.Module):
  def __init__(self, rnn):         # **kwargs is keyword arguments, it is like a dictionary of as many inputs user wants to pass within the dict
    super().__init__()
    self.rnn = rnn
    self.hidden_size = self.rnn.hidden_size
    self.vocab_size = self.rnn.input_size

    if not self.rnn.bidirectional:
      self.num_directions = 1
      self.linear = nn.Linear(self.hidden_size, self.vocab_size) # Here we only define the object shape for the output Layer
    else:
      self.num_directions = 2
      self.linear = nn.Linear(self.hidden_size * 2, self.vocab_size)

  # H: hidden_size; V: input_size/vocab_size; N: batch_size
  def forward(self, inputs, state):
    X = F.one_hot(inputs.T, self.vocab_size) # inputs: [N,T] -> [T,N] -> [T,N,V]
    X = X.to(torch.float32)
    Y, state = self.rnn(X, state) # outputs: [T, N, H]; state: [num_layers, N, H]

    # Linear layers expects 1D input and 1D output with additional batch size
    # So here, we send the 1D input as H and 1D output as V, with additional batch_size as T*N
    # Y.reshape(-1, Y.shape[-1])   # [T, N, H] -> [T*N, H]

    outputs = self.linear(Y.reshape(-1, Y.shape[-1])) # [T*N, V] Here we give the input to the output layer

    return outputs, state # State: [num_layers, N, H]

  def init_state(self, batch_size, device):
    # Default: bidirectional=False so num_directions = 1
    return torch.zeros((self.num_directions * self.rnn.num_layers, batch_size, self.hidden_size), device=device) # state: [num_layers, N, H]


In [30]:
# Prediction without training

net = RNNModel(rnn)
net = net.to(device)

num_preds = 100
predict_ = lambda prefix: predict(net, prefix, num_preds, device)
print(predict_("time traveller"))
print(predict_("the"))

time travellerwcxqcqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjq
thecxwqjqcqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjqjq


In [31]:
train(n_epochs=500,
      lr = 1,
      data_loader = train_iter,
      model = net,
      device = device)

time traveller mone dare wong thepritheramoncruand vitherdithere
time traveller hos the forld upon the evening ofmy arrivglanci b
time traveller potw uling ananded reyous manked any fuss about h
time traveller por the and and his usually pale f cons and how i
time traveller and rabllthe moss ther sastisi nad uitwof the mam
time traveller hom shemin out upon the evening ofmy arrival i th
time traveller the flin the dark a hand touched mine lank finger
time traveller hold shon and the agricuparst wraighifrain was th
time traveller ppas couine ow the day so idecided hou leys we be
time traveller puas fresand a ly explond toouses the evinted of 
time traveller hold but of i remembered motime traveller hold bu
time traveller por that i should explain was the date the little
time traveller por the al man the dark a hand touched mine lank 
time traveller s faceiigl istiellion and the ad with my mind i c
time traveller parassi though it was at my own expense i could n
time traveller put the da

In [32]:
num_preds = 500
predict_ = lambda prefix: predict(net, prefix, num_preds, device)
print(predict_("time traveller"))
print(predict_("the"))

time traveller holden them they spent all their time in playingge and his usually pale face was flushed and animations among them they spent all their time in playingge and his usually pale face was flushed and animations among them they spent all their time in playingge and his usually pale face was flushed and animations among them they spent all their time in playingge and his usually pale face was flushed and animations among them they spent all their time in playingge and his usually pale face was flushe
the heels of that came a strange thing the scietiis of houses the evi had viewed tow metal from which i had viewed the world upon the evening ofmy arrival had ga close on the heels of that came a strange thing the scietich my hand with afrigh as ing hough it was at my own expense i could hear the sphinx and weeping with absoluted any fuss about himself for a minute perhapsfour feena came very closed so loins of underground life and as happy inthe night the stood just behind me its